# Cry Reason Pipeline: Full Embedded Code Notebook

Each section contains the full code from one script directly embedded in this notebook.
Run each code cell to see progress in that cell output.


## 01_dataset_audit.py


In [1]:
print('===== START 01_dataset_audit.py =====')
import argparse
import contextlib
import json
import os
import statistics
import wave
from pathlib import Path

import pandas as pd

ALLOWED_CLASSES = ["belly_pain", "burping", "discomfort", "hungry", "tired"]


def parse_filename_metadata(file_name: str) -> dict:
    stem = Path(file_name).stem
    parts = stem.split("-")

    gender = None
    age_code = None
    reason_code = None

    if len(parts) >= 3:
        gender = parts[-3]
        age_code = parts[-2]
        reason_code = parts[-1]

    return {
        "gender": gender,
        "age_code": age_code,
        "reason_code": reason_code,
    }


def wav_info(path: Path) -> dict:
    with contextlib.closing(wave.open(str(path), "rb")) as wav_reader:
        sample_rate = wav_reader.getframerate()
        num_frames = wav_reader.getnframes()
        channels = wav_reader.getnchannels()
        duration_sec = num_frames / float(sample_rate) if sample_rate else 0.0

    return {
        "sample_rate": sample_rate,
        "channels": channels,
        "duration_sec": duration_sec,
    }


def collect_rows(dataset_root: Path) -> pd.DataFrame:
    rows = []
    for class_name in sorted(ALLOWED_CLASSES):
        class_dir = dataset_root / class_name
        if not class_dir.exists():
            continue

        for wav_path in class_dir.rglob("*.wav"):
            meta = parse_filename_metadata(wav_path.name)
            audio = wav_info(wav_path)
            rows.append(
                {
                    "path": str(wav_path.resolve()),
                    "label": class_name,
                    **meta,
                    **audio,
                }
            )

    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> dict:
    if df.empty:
        return {
            "num_samples": 0,
            "class_counts": {},
            "sample_rates": {},
            "channels": {},
            "duration": {},
        }

    duration_vals = df["duration_sec"].tolist()
    return {
        "num_samples": int(len(df)),
        "class_counts": {k: int(v) for k, v in df["label"].value_counts().sort_index().items()},
        "sample_rates": {str(k): int(v) for k, v in df["sample_rate"].value_counts().sort_index().items()},
        "channels": {str(k): int(v) for k, v in df["channels"].value_counts().sort_index().items()},
        "duration": {
            "min": round(float(min(duration_vals)), 4),
            "max": round(float(max(duration_vals)), 4),
            "mean": round(float(statistics.mean(duration_vals)), 4),
            "median": round(float(statistics.median(duration_vals)), 4),
        },
    }


def main() -> None:
    parser = argparse.ArgumentParser(description="Audit cleaned Donate-a-Cry WAV dataset")
    parser.add_argument(
        "--dataset-root",
        type=Path,
        default=Path("../donateacry-corpus-master/donateacry_corpus_cleaned_and_updated_data"),
        help="Path to cleaned Donate-a-Cry dataset root",
    )
    parser.add_argument(
        "--out-dir",
        type=Path,
        default=Path("./artifacts"),
        help="Directory to write audit files",
    )
    args = parser.parse_args(args=[])

    dataset_root = args.dataset_root.resolve()
    out_dir = args.out_dir.resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    df = collect_rows(dataset_root)
    if df.empty:
        raise RuntimeError(f"No WAV files found under: {dataset_root}")

    df = df.sort_values(["label", "path"]).reset_index(drop=True)

    metadata_csv = out_dir / "metadata.csv"
    summary_json = out_dir / "dataset_summary.json"

    df.to_csv(metadata_csv, index=False)
    with summary_json.open("w", encoding="utf-8") as f:
        json.dump(summarize(df), f, indent=2)

    print(f"Saved metadata: {metadata_csv}")
    print(f"Saved summary:  {summary_json}")


if __name__ == "__main__":
    main()

print('===== END {fname} =====')

===== START 01_dataset_audit.py =====
Saved metadata: D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\metadata.csv
Saved summary:  D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\dataset_summary.json
===== END {fname} =====


## 02_prepare_splits.py


In [2]:
print('===== START 02_prepare_splits.py =====')
import argparse
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


def main() -> None:
    parser = argparse.ArgumentParser(description="Create train/val/test stratified splits")
    parser.add_argument(
        "--metadata-csv",
        type=Path,
        default=Path("./artifacts/metadata.csv"),
        help="Metadata CSV from 01_dataset_audit.py",
    )
    parser.add_argument(
        "--out-dir",
        type=Path,
        default=Path("./artifacts/splits"),
        help="Output directory for split CSV files",
    )
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    args = parser.parse_args(args=[])

    metadata_csv = args.metadata_csv.resolve()
    out_dir = args.out_dir.resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(metadata_csv)
    if "label" not in df.columns:
        raise ValueError("metadata CSV must contain a 'label' column")

    train_df, test_df = train_test_split(
        df,
        test_size=0.15,
        random_state=args.seed,
        stratify=df["label"],
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=0.1764705882,
        random_state=args.seed,
        stratify=train_df["label"],
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    train_path = out_dir / "train.csv"
    val_path = out_dir / "val.csv"
    test_path = out_dir / "test.csv"

    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)

    print(f"Train: {len(train_df)} -> {train_path}")
    print(f"Val:   {len(val_df)} -> {val_path}")
    print(f"Test:  {len(test_df)} -> {test_path}")
    print("Class balance in train split:")
    print(train_df["label"].value_counts().sort_index())


if __name__ == "__main__":
    main()
print('===== END {fname} =====')

===== START 02_prepare_splits.py =====
Train: 319 -> D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\splits\train.csv
Val:   69 -> D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\splits\val.csv
Test:  69 -> D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\splits\test.csv
Class balance in train split:
label
belly_pain     12
burping         6
discomfort     19
hungry        266
tired          16
Name: count, dtype: int64
===== END {fname} =====


## 03_train_model.py


In [3]:
print('===== START 03_train_model.py =====')
import argparse
import json
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight


def load_audio_fixed(path: str, sr: int, clip_seconds: float) -> np.ndarray:
    clip_len = int(sr * clip_seconds)
    wav, _ = librosa.load(path, sr=sr, mono=True)
    if len(wav) < clip_len:
        wav = np.pad(wav, (0, clip_len - len(wav)), mode="constant")
    else:
        wav = wav[:clip_len]
    return wav


def to_log_mel(wav: np.ndarray, sr: int, n_mels: int, n_fft: int, hop_length: int) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=wav,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    return log_mel.astype(np.float32)


def augment_waveform(
    wav: np.ndarray,
    rng: np.random.Generator,
    noise_std: float,
    max_shift_samples: int,
    gain_min: float,
    gain_max: float,
) -> np.ndarray:
    out = wav.copy()

    if max_shift_samples > 0 and rng.random() < 0.5:
        shift = int(rng.integers(-max_shift_samples, max_shift_samples + 1))
        out = np.roll(out, shift)

    if noise_std > 0 and rng.random() < 0.6:
        out = out + rng.normal(0.0, noise_std, size=out.shape).astype(np.float32)

    if gain_max > gain_min and rng.random() < 0.5:
        gain = float(rng.uniform(gain_min, gain_max))
        out = out * gain

    out = np.clip(out, -1.0, 1.0)
    return out


def augment_log_mel(
    feat: np.ndarray,
    rng: np.random.Generator,
    max_time_mask: int,
    max_freq_mask: int,
) -> np.ndarray:
    out = feat.copy()
    n_mels, n_frames = out.shape

    if max_time_mask > 0 and rng.random() < 0.6:
        width = int(rng.integers(0, min(max_time_mask, n_frames) + 1))
        if width > 0:
            start = int(rng.integers(0, n_frames - width + 1))
            out[:, start : start + width] = out.min()

    if max_freq_mask > 0 and rng.random() < 0.6:
        width = int(rng.integers(0, min(max_freq_mask, n_mels) + 1))
        if width > 0:
            start = int(rng.integers(0, n_mels - width + 1))
            out[start : start + width, :] = out.min()

    return out


def build_dataset(
    df: pd.DataFrame,
    label_to_index: dict,
    sr: int,
    clip_seconds: float,
    n_mels: int,
    n_fft: int,
    hop_length: int,
    augment: bool = False,
    minority_ratio: float = 1.0,
    rng: np.random.Generator = None,
    noise_std: float = 0.002,
    max_shift_samples: int = 160,
    gain_min: float = 0.85,
    gain_max: float = 1.15,
    max_time_mask: int = 6,
    max_freq_mask: int = 6,
):
    if rng is None:
        rng = np.random.default_rng(42)

    features = []
    labels = []

    for _, row in df.iterrows():
        row_ratio = float(row.get("_minority_ratio", minority_ratio))
        wav = load_audio_fixed(row["path"], sr=sr, clip_seconds=clip_seconds)
        if augment and rng.random() < row_ratio:
            wav = augment_waveform(
                wav,
                rng=rng,
                noise_std=noise_std,
                max_shift_samples=max_shift_samples,
                gain_min=gain_min,
                gain_max=gain_max,
            )

        feat = to_log_mel(wav, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length)

        if augment and rng.random() < row_ratio:
            feat = augment_log_mel(
                feat,
                rng=rng,
                max_time_mask=max_time_mask,
                max_freq_mask=max_freq_mask,
            )

        features.append(feat)
        labels.append(label_to_index[row["label"]])

    x = np.stack(features, axis=0)
    x = x[..., np.newaxis]

    mean = x.mean()
    std = x.std() + 1e-6
    x = (x - mean) / std

    y = np.array(labels, dtype=np.int32)
    return x.astype(np.float32), y, float(mean), float(std)


def make_balanced_train_df(train_df: pd.DataFrame, seed: int) -> pd.DataFrame:
    counts = train_df["label"].value_counts()
    target = int(counts.max())
    rng = np.random.default_rng(seed)

    frames = []
    for label, group in train_df.groupby("label"):
        idx = rng.choice(group.index.to_numpy(), size=target, replace=True)
        sampled = train_df.loc[idx]
        sampled = sampled.assign(_source_count=int(counts[label]))
        frames.append(sampled)

    balanced = pd.concat(frames, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return balanced


def create_focal_loss(alpha=0.25, gamma=2.0):
    def focal_loss(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        y_true_one_hot = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1], dtype=y_pred.dtype)

        pt = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1)
        ce_loss = -tf.math.log(pt)
        focal_weight = tf.pow(1.0 - pt, gamma)
        return tf.reduce_mean(alpha * focal_weight * ce_loss)
    
    return focal_loss


def create_model(input_shape, num_classes: int, use_focal_loss: bool = False) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=input_shape)

    x = tf.keras.layers.Conv2D(16, (3, 3), padding="same", activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D((2, 2))(x)

    x = tf.keras.layers.SeparableConv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D((2, 2))(x)

    x = tf.keras.layers.SeparableConv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    loss_fn = create_focal_loss() if use_focal_loss else "sparse_categorical_crossentropy"
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=loss_fn,
        metrics=["accuracy"],
    )
    return model


def main() -> None:
    parser = argparse.ArgumentParser(description="Train baby cry reason classifier")
    parser.add_argument("--splits-dir", type=Path, default=Path("./artifacts/splits"))
    parser.add_argument("--out-dir", type=Path, default=Path("./artifacts/model"))
    parser.add_argument("--sample-rate", type=int, default=8000)
    parser.add_argument("--clip-seconds", type=float, default=7.0)
    parser.add_argument("--n-mels", type=int, default=40)
    parser.add_argument("--n-fft", type=int, default=512)
    parser.add_argument("--hop-length", type=int, default=160)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--early-stop-patience", type=int, default=20)
    parser.add_argument("--lr-plateau-patience", type=int, default=4)
    parser.add_argument("--balanced-sampling", action="store_true")
    parser.add_argument("--augment-minority", action="store_true")
    parser.add_argument("--aug-noise-std", type=float, default=0.002)
    parser.add_argument("--aug-time-shift-max", type=float, default=0.02)
    parser.add_argument("--aug-gain-min", type=float, default=0.85)
    parser.add_argument("--aug-gain-max", type=float, default=1.15)
    parser.add_argument("--aug-time-mask-max", type=int, default=6)
    parser.add_argument("--aug-freq-mask-max", type=int, default=6)
    parser.add_argument("--use-focal-loss", action="store_true", help="Use focal loss for better minority handling")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args(args=[])

    tf.keras.utils.set_random_seed(args.seed)

    out_dir = args.out_dir.resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    train_df = pd.read_csv((args.splits_dir / "train.csv").resolve())
    val_df = pd.read_csv((args.splits_dir / "val.csv").resolve())
    test_df = pd.read_csv((args.splits_dir / "test.csv").resolve())

    train_counts = train_df["label"].value_counts()

    labels_sorted = sorted(train_df["label"].unique().tolist())
    label_to_index = {label: idx for idx, label in enumerate(labels_sorted)}
    index_to_label = {idx: label for label, idx in label_to_index.items()}

    train_df_used = train_df
    if args.balanced_sampling:
        train_df_used = make_balanced_train_df(train_df, seed=args.seed)

    max_count = int(train_counts.max())
    train_df_used = train_df_used.copy()
    train_df_used["_minority_ratio"] = train_df_used["label"].map(
        lambda x: min(1.0, max_count / float(train_counts[x] * 4.0))
    )

    rng = np.random.default_rng(args.seed)
    max_shift_samples = int(max(0.0, args.aug_time_shift_max) * args.sample_rate)

    x_train, y_train, mean, std = build_dataset(
        train_df_used,
        label_to_index,
        sr=args.sample_rate,
        clip_seconds=args.clip_seconds,
        n_mels=args.n_mels,
        n_fft=args.n_fft,
        hop_length=args.hop_length,
        augment=args.augment_minority,
        minority_ratio=1.0,
        rng=rng,
        noise_std=args.aug_noise_std,
        max_shift_samples=max_shift_samples,
        gain_min=args.aug_gain_min,
        gain_max=args.aug_gain_max,
        max_time_mask=args.aug_time_mask_max,
        max_freq_mask=args.aug_freq_mask_max,
    )
    x_val, y_val, _, _ = build_dataset(
        val_df,
        label_to_index,
        sr=args.sample_rate,
        clip_seconds=args.clip_seconds,
        n_mels=args.n_mels,
        n_fft=args.n_fft,
        hop_length=args.hop_length,
    )
    x_test, y_test, _, _ = build_dataset(
        test_df,
        label_to_index,
        sr=args.sample_rate,
        clip_seconds=args.clip_seconds,
        n_mels=args.n_mels,
        n_fft=args.n_fft,
        hop_length=args.hop_length,
    )

    classes = np.array(sorted(np.unique(y_train)))
    class_weights_arr = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train,
    )
    class_weights = {int(cls): float(wt) for cls, wt in zip(classes, class_weights_arr)}
    
    if args.use_focal_loss:
        class_weights = {int(cls): 1.0 for cls in classes}

    model = create_model(input_shape=x_train.shape[1:], num_classes=len(labels_sorted), use_focal_loss=args.use_focal_loss)

    tmp_best_model_path = out_dir / "_tmp_best_model.keras"
    if tmp_best_model_path.exists():
        tmp_best_model_path.unlink()

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(tmp_best_model_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            save_weights_only=False,
            verbose=0,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=args.early_stop_patience,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=args.lr_plateau_patience,
            min_lr=1e-5,
        ),
    ]

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=args.epochs,
        batch_size=args.batch_size,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=2,
    )

    if tmp_best_model_path.exists():
        model = tf.keras.models.load_model(tmp_best_model_path)

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    y_pred_proba_test = model.predict(x_test, verbose=0)
    y_pred = np.argmax(y_pred_proba_test, axis=1)
    
    # Also get validation predictions for threshold optimization
    y_pred_proba_val = model.predict(x_val, verbose=0)
    
    # Save predictions and probabilities for threshold optimization
    np.save(out_dir / "val_predictions.npy", y_pred_proba_val)
    np.save(out_dir / "val_labels.npy", y_val)
    np.save(out_dir / "test_predictions.npy", y_pred_proba_test)
    np.save(out_dir / "test_labels.npy", y_test)

    report = classification_report(
        y_test,
        y_pred,
        output_dict=True,
        target_names=labels_sorted,
        zero_division=0,
    )
    conf = confusion_matrix(y_test, y_pred).tolist()

    keras_model_path = out_dir / "cry_reason_model.keras"
    if keras_model_path.exists():
        keras_model_path.unlink()
    model.save(keras_model_path)

    if tmp_best_model_path.exists():
        tmp_best_model_path.unlink()

    np.save(out_dir / "x_train_features.npy", x_train.astype(np.float32))

    training_info = {
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "label_to_index": label_to_index,
        "index_to_label": {str(k): v for k, v in index_to_label.items()},
        "class_weights": {str(k): v for k, v in class_weights.items()},
        "normalization": {"mean": mean, "std": std},
        "feature_params": {
            "sample_rate": args.sample_rate,
            "clip_seconds": args.clip_seconds,
            "n_mels": args.n_mels,
            "n_fft": args.n_fft,
            "hop_length": args.hop_length,
        },
        "classification_report": report,
        "confusion_matrix": conf,
        "history": {k: [float(vv) for vv in vals] for k, vals in history.history.items()},
    }

    with (out_dir / "training_info.json").open("w", encoding="utf-8") as f:
        json.dump(training_info, f, indent=2)
    
    # Save metadata for threshold optimization
    metadata = {
        "labels_sorted": labels_sorted,
        "label_to_index": label_to_index,
        "index_to_label": {str(k): v for k, v in index_to_label.items()},
    }
    with (out_dir / "metadata.json").open("w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    per_class_rows = []
    for label in labels_sorted:
        item = report.get(label, {})
        per_class_rows.append(
            {
                "label": label,
                "precision": float(item.get("precision", 0.0)),
                "recall": float(item.get("recall", 0.0)),
                "f1": float(item.get("f1-score", 0.0)),
                "support": int(item.get("support", 0)),
            }
        )
    pd.DataFrame(per_class_rows).to_csv(out_dir / "per_class_metrics.csv", index=False)

    print(f"Saved model: {keras_model_path}")
    print(f"Saved per-class metrics: {out_dir / 'per_class_metrics.csv'}")
    print(f"Test accuracy: {test_acc:.4f}")


if __name__ == "__main__":
    main()
print('===== END {fname} =====')

===== START 03_train_model.py =====



Epoch 1/100


20/20 - 7s - loss: 1.6330 - accuracy: 0.2759 - val_loss: 1.6089 - val_accuracy: 0.1739 - lr: 0.0010 - 7s/epoch - 358ms/step
Epoch 2/100
20/20 - 1s - loss: 1.5259 - accuracy: 0.2539 - val_loss: 1.6276 - val_accuracy: 0.0435 - lr: 0.0010 - 1s/epoch - 61ms/step
Epoch 3/100
20/20 - 2s - loss: 1.5713 - accuracy: 0.2414 - val_loss: 1.6405 - val_accuracy: 0.0290 - lr: 0.0010 - 2s/epoch - 79ms/step
Epoch 4/100
20/20 - 2s - loss: 1.5533 - accuracy: 0.2665 - val_loss: 1.6554 - val_accuracy: 0.0290 - lr: 0.0010 - 2s/epoch - 90ms/step
Epoch 5/100
20/20 - 2s - loss: 1.5188 - accuracy: 0.2978 - val_loss: 1.6747 - val_accuracy: 0.0435 - lr: 0.0010 - 2s/epoch - 81ms/step
Epoch 6/100
20/20 - 1s - loss: 1.5679 - accuracy: 0.2351 - val_loss: 1.6898 - val_accuracy: 0.0290 - lr: 5.0000e-04 - 1s/epoch - 70ms/step
Epoch 7/100
20/20 - 1s - loss: 1.4502 - accuracy: 0.2351 - val_loss: 1.7187 - val_accuracy: 0.0290 - lr: 5.0000e-04 - 1s/epoch - 66ms/step
Epoch

## 04_export_tflite.py


In [4]:
print('===== START 04_export_tflite.py =====')
import argparse
from pathlib import Path

import numpy as np
import tensorflow as tf


def representative_dataset(features: np.ndarray, max_samples: int = 100):
    count = min(len(features), max_samples)
    for i in range(count):
        sample = features[i : i + 1].astype(np.float32)
        yield [sample]


def main() -> None:
    parser = argparse.ArgumentParser(description="Export trained Keras model to TFLite")
    parser.add_argument("--model-dir", type=Path, default=Path("./artifacts/model"))
    parser.add_argument("--quantize", action="store_true", help="Enable full int8 quantization")
    args = parser.parse_args(args=[])

    model_dir = args.model_dir.resolve()
    keras_model_path = model_dir / "cry_reason_model.keras"
    tflite_path = model_dir / "cry_reason_model.tflite"

    model = tf.keras.models.load_model(keras_model_path)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if args.quantize:
        features_path = model_dir / "x_train_features.npy"
        if not features_path.exists():
            raise FileNotFoundError(
                "x_train_features.npy is required for int8 quantization. Run training first."
            )

        features = np.load(features_path)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_dataset(features)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    tflite_path.write_bytes(tflite_model)

    size_kb = len(tflite_model) / 1024.0
    print(f"Saved TFLite model: {tflite_path}")
    print(f"Model size: {size_kb:.2f} KB")


if __name__ == "__main__":
    main()
print('===== END {fname} =====')

===== START 04_export_tflite.py =====
INFO:tensorflow:Assets written to: C:\Users\Dell\AppData\Local\Temp\tmp_iyfgyy3\assets


INFO:tensorflow:Assets written to: C:\Users\Dell\AppData\Local\Temp\tmp_iyfgyy3\assets


Saved TFLite model: D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\model\cry_reason_model.tflite
Model size: 21.24 KB
===== END {fname} =====


## 05_infer_clip.py


In [7]:
print('===== START 05_infer_clip.py =====')
import argparse
import json
from pathlib import Path
from typing import Optional

import librosa
import numpy as np
import tensorflow as tf


def preprocess_clip(path: Path, feature_params: dict, norm: dict) -> np.ndarray:
    sr = int(feature_params["sample_rate"])
    clip_seconds = float(feature_params["clip_seconds"])
    n_mels = int(feature_params["n_mels"])
    n_fft = int(feature_params["n_fft"])
    hop_length = int(feature_params["hop_length"])

    clip_len = int(sr * clip_seconds)
    wav, _ = librosa.load(str(path), sr=sr, mono=True)

    if len(wav) < clip_len:
        wav = np.pad(wav, (0, clip_len - len(wav)), mode="constant")
    else:
        wav = wav[:clip_len]

    mel = librosa.feature.melspectrogram(
        y=wav,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0,
    )
    feat = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
    feat = feat[..., np.newaxis]

    mean = float(norm["mean"])
    std = float(norm["std"]) + 1e-6
    feat = (feat - mean) / std

    return feat[np.newaxis, ...].astype(np.float32)


def predict_tflite(interpreter: tf.lite.Interpreter, x: np.ndarray) -> np.ndarray:
    interpreter.allocate_tensors()
    in_detail = interpreter.get_input_details()[0]
    out_detail = interpreter.get_output_details()[0]

    x_input = x
    if in_detail["dtype"] == np.int8:
        scale, zero_point = in_detail["quantization"]
        x_input = (x / scale + zero_point).round().astype(np.int8)

    interpreter.set_tensor(in_detail["index"], x_input)
    interpreter.invoke()
    out = interpreter.get_tensor(out_detail["index"])

    if out_detail["dtype"] == np.int8:
        scale, zero_point = out_detail["quantization"]
        out = scale * (out.astype(np.float32) - zero_point)

    return out.astype(np.float32)


def load_threshold_config(config_path: Path):
    if not config_path.exists():
        return None

    return json.loads(config_path.read_text(encoding="utf-8"))


def apply_thresholds(probs: np.ndarray, threshold_config: Optional[dict]) -> int:
    if threshold_config is None:
        return int(np.argmax(probs))

    if threshold_config.get("type") == "per-class":
        thresholds = {int(k): float(v) for k, v in threshold_config.get("thresholds", {}).items()}
        ordered_classes = np.argsort(probs)[::-1]

        for class_idx in ordered_classes:
            class_idx = int(class_idx)
            if probs[class_idx] >= thresholds.get(class_idx, 0.5):
                return class_idx

    return int(np.argmax(probs))


def resolve_clip_path(clip_arg: Optional[Path]) -> Path:
    if clip_arg is not None:
        clip_path = clip_arg.expanduser().resolve()
        if clip_path.exists():
            return clip_path
        raise FileNotFoundError(f"Clip not found: {clip_path}")

    dataset_roots = [
        Path("../donateacry-corpus-master/donateacry_corpus_cleaned_and_updated_data"),
        Path("./donateacry-corpus-master/donateacry_corpus_cleaned_and_updated_data"),
        Path("../../donateacry-corpus-master/donateacry_corpus_cleaned_and_updated_data"),
    ]

    for root in dataset_roots:
        resolved_root = root.resolve()
        if not resolved_root.exists():
            continue
        wav_files = sorted(resolved_root.rglob("*.wav"))
        if wav_files:
            print(f"No --clip provided. Using sample clip: {wav_files[0]}")
            return wav_files[0]

    searched = [str(p.resolve()) for p in dataset_roots]
    raise FileNotFoundError(
        "No clip provided and no WAV files found in expected dataset roots:\n"
        + "\n".join(searched)
    )


def main() -> None:
    parser = argparse.ArgumentParser(description="Run single-clip inference")
    parser.add_argument("--clip", type=Path, default=None, help="Path to WAV clip (optional in notebook)")
    parser.add_argument("--model-dir", type=Path, default=Path("./artifacts/model"))
    parser.add_argument("--use-tflite", action="store_true", help="Use TFLite model")
    parser.add_argument(
        "--threshold-config",
        type=Path,
        default=Path("./artifacts/model/threshold_config.json"),
        help="Optional threshold configuration JSON",
    )
    args = parser.parse_args(args=[])

    model_dir = args.model_dir.resolve()
    info_path = model_dir / "training_info.json"
    info = json.loads(info_path.read_text(encoding="utf-8"))
    threshold_config = load_threshold_config(args.threshold_config.resolve())

    clip_path = resolve_clip_path(args.clip)
    x = preprocess_clip(clip_path, info["feature_params"], info["normalization"])

    index_to_label = {int(k): v for k, v in info["index_to_label"].items()}

    if args.use_tflite:
        tflite_path = model_dir / "cry_reason_model.tflite"
        interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
        probs = predict_tflite(interpreter, x)[0]
    else:
        keras_path = model_dir / "cry_reason_model.keras"
        model = tf.keras.models.load_model(keras_path)
        probs = model.predict(x, verbose=0)[0]

    top_idx = apply_thresholds(np.asarray(probs, dtype=np.float32), threshold_config)

    print(f"Clip used: {clip_path}")
    print("Prediction:")
    print(f"label={index_to_label[top_idx]} prob={probs[top_idx]:.4f}")
    print("All class probabilities:")
    for idx, prob in enumerate(probs):
        print(f"{index_to_label[idx]}: {prob:.4f}")


if __name__ == "__main__":
    main()
print('===== END {fname} =====')

===== START 05_infer_clip.py =====
No --clip provided. Using sample clip: D:\IISc\sem2\EdgeAI\Project\proj-v2\donateacry-corpus-master\donateacry_corpus_cleaned_and_updated_data\belly_pain\549a46d8-9c84-430e-ade8-97eae2bef787-1430130772174-1.7-m-48-bp.wav
Clip used: D:\IISc\sem2\EdgeAI\Project\proj-v2\donateacry-corpus-master\donateacry_corpus_cleaned_and_updated_data\belly_pain\549a46d8-9c84-430e-ade8-97eae2bef787-1430130772174-1.7-m-48-bp.wav
Prediction:
label=burping prob=0.2026
All class probabilities:
belly_pain: 0.2002
burping: 0.2026
discomfort: 0.2009
hungry: 0.2019
tired: 0.1945
===== END {fname} =====


## 06_run_pipeline.py


In [9]:
print('===== START 06_run_pipeline.py =====')
import argparse
import subprocess
import sys
from pathlib import Path

import pandas as pd


def run_step(command):
    print("\n[RUN]", " ".join(str(c) for c in command))
    subprocess.run(command, check=True)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Run full baby-cry pipeline in sequence with configurable hyperparameters"
    )

    parser.add_argument(
        "--python-exe",
        type=str,
        default=sys.executable,
        help="Python executable to use for running pipeline steps",
    )

    parser.add_argument(
        "--dataset-root",
        type=Path,
        default=Path("../donateacry-corpus-master/donateacry_corpus_cleaned_and_updated_data"),
    )
    parser.add_argument("--artifacts-dir", type=Path, default=Path("./artifacts"))

    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--sample-rate", type=int, default=8000)
    parser.add_argument("--clip-seconds", type=float, default=7.0)
    parser.add_argument("--n-mels", type=int, default=40)
    parser.add_argument("--n-fft", type=int, default=512)
    parser.add_argument("--hop-length", type=int, default=160)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--early-stop-patience", type=int, default=20)
    parser.add_argument("--lr-plateau-patience", type=int, default=4)
    parser.add_argument("--balanced-sampling", action="store_true")
    parser.add_argument("--augment-minority", action="store_true")
    parser.add_argument("--aug-noise-std", type=float, default=0.002)
    parser.add_argument("--aug-time-shift-max", type=float, default=0.02)
    parser.add_argument("--aug-gain-min", type=float, default=0.85)
    parser.add_argument("--aug-gain-max", type=float, default=1.15)
    parser.add_argument("--aug-time-mask-max", type=int, default=6)
    parser.add_argument("--aug-freq-mask-max", type=int, default=6)
    parser.add_argument(
        "--optimize-thresholds",
        action="store_true",
        help="Optimize per-class thresholds after training and export",
    )
    parser.add_argument(
        "--threshold-metric",
        type=str,
        default="f1-macro",
        choices=["f1-weighted", "f1-macro", "balanced-accuracy", "f1-micro"],
        help="Metric to optimize when threshold tuning is enabled",
    )

    parser.add_argument("--quantize", action="store_true", help="Export int8 TFLite")
    parser.add_argument(
        "--skip-audit",
        action="store_true",
        help="Skip dataset audit and metadata generation",
    )
    parser.add_argument(
        "--skip-splits",
        action="store_true",
        help="Skip split generation and reuse existing splits",
    )
    parser.add_argument(
        "--skip-smoke-infer",
        action="store_true",
        help="Skip one-sample TFLite smoke inference",
    )

    args = parser.parse_args(args=[])

    if "__file__" in globals():
        root = Path(__file__).resolve().parent
    else:
        root = Path.cwd().resolve()

    artifacts_dir = (root / args.artifacts_dir).resolve()
    model_dir = artifacts_dir / "model"
    splits_dir = artifacts_dir / "splits"

    py = args.python_exe

    if not args.skip_audit:
        run_step(
            [
                py,
                str(root / "01_dataset_audit.py"),
                "--dataset-root",
                str((root / args.dataset_root).resolve()),
                "--out-dir",
                str(artifacts_dir),
            ]
        )

    if not args.skip_splits:
        run_step(
            [
                py,
                str(root / "02_prepare_splits.py"),
                "--metadata-csv",
                str(artifacts_dir / "metadata.csv"),
                "--out-dir",
                str(splits_dir),
                "--seed",
                str(args.seed),
            ]
        )

    run_step(
        [
            py,
            str(root / "03_train_model.py"),
            "--splits-dir",
            str(splits_dir),
            "--out-dir",
            str(model_dir),
            "--sample-rate",
            str(args.sample_rate),
            "--clip-seconds",
            str(args.clip_seconds),
            "--n-mels",
            str(args.n_mels),
            "--n-fft",
            str(args.n_fft),
            "--hop-length",
            str(args.hop_length),
            "--batch-size",
            str(args.batch_size),
            "--epochs",
            str(args.epochs),
            "--early-stop-patience",
            str(args.early_stop_patience),
            "--lr-plateau-patience",
            str(args.lr_plateau_patience),
            "--aug-noise-std",
            str(args.aug_noise_std),
            "--aug-time-shift-max",
            str(args.aug_time_shift_max),
            "--aug-gain-min",
            str(args.aug_gain_min),
            "--aug-gain-max",
            str(args.aug_gain_max),
            "--aug-time-mask-max",
            str(args.aug_time_mask_max),
            "--aug-freq-mask-max",
            str(args.aug_freq_mask_max),
            "--seed",
            str(args.seed),
        ]
        + (["--balanced-sampling"] if args.balanced_sampling else [])
        + (["--augment-minority"] if args.augment_minority else [])
    )

    export_cmd = [py, str(root / "04_export_tflite.py"), "--model-dir", str(model_dir)]
    if args.quantize:
        export_cmd.append("--quantize")
    run_step(export_cmd)

    if args.optimize_thresholds:
        run_step(
            [
                py,
                str(root / "08_threshold_optimization.py"),
                "--model-dir",
                str(model_dir),
                "--metric",
                str(args.threshold_metric),
                "--output-file",
                str(model_dir / "threshold_config.json"),
            ]
        )

    if not args.skip_smoke_infer:
        test_csv = pd.read_csv(splits_dir / "test.csv")
        if len(test_csv) > 0:
            sample_path = str(test_csv.iloc[0]["path"])
            run_step(
                [
                    py,
                    str(root / "05_infer_clip.py"),
                    "--use-tflite",
                    "--model-dir",
                    str(model_dir),
                    "--clip",
                    sample_path,
                ]
            )

    print("Pipeline completed successfully.")
    print(f"Best model: {model_dir / 'cry_reason_model.keras'}")
    print(f"Per-class F1: {model_dir / 'per_class_metrics.csv'}")
    print(f"TFLite model: {model_dir / 'cry_reason_model.tflite'}")


if __name__ == "__main__":
    main()
print('===== END {fname} =====')

===== START 06_run_pipeline.py =====

[RUN] d:\IISc\sem2\EdgeAI\Project\.venv\Scripts\python.exe D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\01_dataset_audit.py --dataset-root D:\IISc\sem2\EdgeAI\Project\proj-v2\donateacry-corpus-master\donateacry_corpus_cleaned_and_updated_data --out-dir D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts

[RUN] d:\IISc\sem2\EdgeAI\Project\.venv\Scripts\python.exe D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\02_prepare_splits.py --metadata-csv D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\metadata.csv --out-dir D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\splits --seed 42

[RUN] d:\IISc\sem2\EdgeAI\Project\.venv\Scripts\python.exe D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\03_train_model.py --splits-dir D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\splits --out-dir D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\model --samp

## 07_sweep_augmentation.py


In [14]:
print('===== START 07_sweep_augmentation.py =====')
import argparse
import json
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd


def run_cmd(cmd):
    print("[RUN]", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, check=True)


def macro_f1(per_class_csv: Path) -> float:
    df = pd.read_csv(per_class_csv)
    return float(df["f1"].mean())


def main() -> None:
    parser = argparse.ArgumentParser(description="Sweep augmentation hyperparameters and pick best macro-F1")
    parser.add_argument("--python-exe", type=str, default=sys.executable)
    parser.add_argument("--splits-dir", type=Path, default=Path("./artifacts/splits"))
    parser.add_argument("--sweep-root", type=Path, default=Path("./artifacts/sweeps"))
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--sample-rate", type=int, default=8000)
    parser.add_argument("--clip-seconds", type=float, default=7.0)
    parser.add_argument("--n-mels", type=int, default=40)
    parser.add_argument("--n-fft", type=int, default=512)
    parser.add_argument("--hop-length", type=int, default=160)
    parser.add_argument("--early-stop-patience", type=int, default=20)
    parser.add_argument("--lr-plateau-patience", type=int, default=5)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args(args=[])

    if "__file__" in globals():
        root = Path(__file__).resolve().parent
    else:
        root = Path.cwd().resolve()

    py = args.python_exe

    splits_dir = (root / args.splits_dir).resolve()
    sweep_root = (root / args.sweep_root).resolve()
    sweep_root.mkdir(parents=True, exist_ok=True)

    candidates = [
        {
            "name": "aug_a",
            "aug_noise_std": 0.002,
            "aug_time_shift_max": 0.02,
            "aug_gain_min": 0.85,
            "aug_gain_max": 1.15,
            "aug_time_mask_max": 6,
            "aug_freq_mask_max": 6,
        },
        {
            "name": "aug_b",
            "aug_noise_std": 0.003,
            "aug_time_shift_max": 0.03,
            "aug_gain_min": 0.8,
            "aug_gain_max": 1.2,
            "aug_time_mask_max": 8,
            "aug_freq_mask_max": 8,
        },
        {
            "name": "aug_c",
            "aug_noise_std": 0.0015,
            "aug_time_shift_max": 0.015,
            "aug_gain_min": 0.9,
            "aug_gain_max": 1.1,
            "aug_time_mask_max": 5,
            "aug_freq_mask_max": 5,
        },
        {
            "name": "aug_d",
            "aug_noise_std": 0.004,
            "aug_time_shift_max": 0.04,
            "aug_gain_min": 0.75,
            "aug_gain_max": 1.25,
            "aug_time_mask_max": 10,
            "aug_freq_mask_max": 10,
        },
    ]

    results = []

    for cfg in candidates:
        run_dir = sweep_root / cfg["name"]
        model_dir = run_dir / "model"
        run_dir.mkdir(parents=True, exist_ok=True)

        train_cmd = [
            py,
            str(root / "03_train_model.py"),
            "--splits-dir",
            str(splits_dir),
            "--out-dir",
            str(model_dir),
            "--sample-rate",
            str(args.sample_rate),
            "--clip-seconds",
            str(args.clip_seconds),
            "--n-mels",
            str(args.n_mels),
            "--n-fft",
            str(args.n_fft),
            "--hop-length",
            str(args.hop_length),
            "--batch-size",
            str(args.batch_size),
            "--epochs",
            str(args.epochs),
            "--early-stop-patience",
            str(args.early_stop_patience),
            "--lr-plateau-patience",
            str(args.lr_plateau_patience),
            "--aug-noise-std",
            str(cfg["aug_noise_std"]),
            "--aug-time-shift-max",
            str(cfg["aug_time_shift_max"]),
            "--aug-gain-min",
            str(cfg["aug_gain_min"]),
            "--aug-gain-max",
            str(cfg["aug_gain_max"]),
            "--aug-time-mask-max",
            str(cfg["aug_time_mask_max"]),
            "--aug-freq-mask-max",
            str(cfg["aug_freq_mask_max"]),
            "--seed",
            str(args.seed),
            "--balanced-sampling",
            "--augment-minority",
        ]
        run_cmd(train_cmd)

        export_cmd = [
            py,
            str(root / "04_export_tflite.py"),
            "--model-dir",
            str(model_dir),
            "--quantize",
        ]
        run_cmd(export_cmd)

        per_class_csv = model_dir / "per_class_metrics.csv"
        info_json = model_dir / "training_info.json"

        mf1 = macro_f1(per_class_csv)
        info = json.loads(info_json.read_text(encoding="utf-8"))
        results.append(
            {
                "name": cfg["name"],
                "macro_f1": mf1,
                "test_accuracy": float(info["test_accuracy"]),
                **cfg,
                "run_dir": str(run_dir),
            }
        )

    results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
    summary_csv = sweep_root / "sweep_results.csv"
    results_df.to_csv(summary_csv, index=False)

    best = results_df.iloc[0].to_dict()
    best_dir = Path(str(best["run_dir"])) / "model"

    final_model_dir = (root / "artifacts/model").resolve()
    final_model_dir.mkdir(parents=True, exist_ok=True)

    for name in ["cry_reason_model.keras", "cry_reason_model.tflite", "training_info.json", "per_class_metrics.csv", "x_train_features.npy"]:
        src = best_dir / name
        dst = final_model_dir / name
        if src.exists():
            shutil.copy2(src, dst)

    best_json = sweep_root / "best_config.json"
    best_json.write_text(json.dumps(best, indent=2), encoding="utf-8")

    print("Sweep complete")
    print(f"Summary: {summary_csv}")
    print(f"Best config: {best_json}")
    print(f"Best macro-F1: {best['macro_f1']:.4f}")
    print(f"Promoted model dir: {final_model_dir}")


if __name__ == "__main__":
    main()
print('===== END {fname} =====')

===== START 07_sweep_augmentation.py =====
[RUN] d:\IISc\sem2\EdgeAI\Project\.venv\Scripts\python.exe D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\03_train_model.py --splits-dir D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\splits --out-dir D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\sweeps\aug_a\model --sample-rate 8000 --clip-seconds 7.0 --n-mels 40 --n-fft 512 --hop-length 160 --batch-size 16 --epochs 100 --early-stop-patience 20 --lr-plateau-patience 5 --aug-noise-std 0.002 --aug-time-shift-max 0.02 --aug-gain-min 0.85 --aug-gain-max 1.15 --aug-time-mask-max 6 --aug-freq-mask-max 6 --seed 42 --balanced-sampling --augment-minority
[RUN] d:\IISc\sem2\EdgeAI\Project\.venv\Scripts\python.exe D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\04_export_tflite.py --model-dir D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\sweeps\aug_a\model --quantize
[RUN] d:\IISc\sem2\EdgeAI\Project\.venv\Scripts\python.exe D:\

## 08_threshold_optimization.py


In [15]:
print('===== START 08_threshold_optimization.py =====')
#!/usr/bin/env python3
"""
Threshold Optimization for Imbalanced Classification.

Finds per-class probability thresholds on the validation set and writes them
to a JSON config that can be consumed by inference.
"""

import argparse
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score, precision_recall_curve


def compute_metrics(y_true, y_pred, metric="f1-weighted"):
    if metric == "f1-weighted":
        return f1_score(y_true, y_pred, average="weighted", zero_division=0)
    if metric == "f1-macro":
        return f1_score(y_true, y_pred, average="macro", zero_division=0)
    if metric == "balanced-accuracy":
        return balanced_accuracy_score(y_true, y_pred)
    if metric == "f1-micro":
        return f1_score(y_true, y_pred, average="micro", zero_division=0)
    raise ValueError(f"Unknown metric: {metric}")


def apply_threshold(y_proba, threshold=0.5, class_thresholds=None):
    if class_thresholds is None:
        return np.argmax(y_proba, axis=1)

    y_pred = np.zeros(len(y_proba), dtype=np.int32)
    for i, proba_sample in enumerate(y_proba):
        ordered_classes = np.argsort(proba_sample)[::-1]
        chosen_class = int(ordered_classes[0])

        for class_idx in ordered_classes:
            class_idx = int(class_idx)
            class_thresh = float(class_thresholds.get(class_idx, threshold))
            if proba_sample[class_idx] >= class_thresh:
                chosen_class = class_idx
                break

        y_pred[i] = chosen_class

    return y_pred


def find_per_class_thresholds(y_true, y_proba, n_classes, labels_sorted, metric="f1-weighted"):
    class_thresholds = {}

    for class_idx in range(n_classes):
        binary_true = (y_true == class_idx).astype(np.int32)
        precision, recall, thresholds = precision_recall_curve(binary_true, y_proba[:, class_idx])

        if len(thresholds) == 0:
            class_thresholds[class_idx] = 0.5
            continue

        precision = precision[:-1]
        recall = recall[:-1]
        f1_values = np.where(
            (precision + recall) > 0,
            (2.0 * precision * recall) / (precision + recall),
            0.0,
        )

        best_idx = int(np.argmax(f1_values))
        class_thresholds[class_idx] = float(thresholds[best_idx])

    y_pred_final = apply_threshold(y_proba, class_thresholds=class_thresholds)
    final_score = compute_metrics(y_true, y_pred_final, metric)
    return class_thresholds, final_score


def load_labels_sorted(model_dir: Path, data_dir: Path):
    candidates = [
        model_dir / "metadata.json",
        data_dir / "metadata.json",
        model_dir / "training_info.json",
        data_dir / "training_info.json",
    ]

    for candidate in candidates:
        if not candidate.exists():
            continue

        with candidate.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        if "labels_sorted" in payload:
            return payload["labels_sorted"]

        if "index_to_label" in payload:
            index_to_label = payload["index_to_label"]
            return [index_to_label[str(idx)] for idx in sorted(map(int, index_to_label.keys()))]

    raise FileNotFoundError(
        "Could not find labels metadata in model-dir or data-dir. Expected metadata.json or training_info.json."
    )


def main():
    parser = argparse.ArgumentParser(description="Optimize thresholds for imbalanced classification")
    parser.add_argument(
        "--model-dir",
        type=str,
        default="artifacts/model",
        help="Directory containing trained model artifacts",
    )
    parser.add_argument(
        "--data-dir",
        type=str,
        default="artifacts/data",
        help="Directory containing data metadata if not present in model-dir",
    )
    parser.add_argument(
        "--metric",
        type=str,
        default="f1-macro",
        choices=["f1-weighted", "f1-macro", "balanced-accuracy", "f1-micro"],
        help="Metric to optimize",
    )
    parser.add_argument(
        "--output-file",
        type=str,
        default="artifacts/model/threshold_config.json",
        help="Output file for threshold configuration",
    )
    args = parser.parse_args(args=[])

    model_dir = Path(args.model_dir)
    data_dir = Path(args.data_dir)

    labels_sorted = load_labels_sorted(model_dir, data_dir)
    n_classes = len(labels_sorted)

    val_predictions_path = model_dir / "val_predictions.npy"
    val_labels_path = model_dir / "val_labels.npy"
    if not val_predictions_path.exists() or not val_labels_path.exists():
        raise FileNotFoundError(
            "Validation predictions not found. Run training first so val_predictions.npy and val_labels.npy are saved in the model directory."
        )

    val_predictions = np.load(val_predictions_path)
    val_labels = np.load(val_labels_path)

    print(f"Loaded {len(val_labels)} validation samples with {n_classes} classes")
    print(f"Optimizing for: {args.metric}")
    print("\nFinding per-class optimal thresholds...")

    class_thresholds, best_score = find_per_class_thresholds(
        val_labels, val_predictions, n_classes, labels_sorted, args.metric
    )

    print("\nOptimal per-class thresholds:")
    for class_idx, thresh in sorted(class_thresholds.items()):
        print(f"  {labels_sorted[class_idx]}: {thresh:.3f}")

    print(f"\nBest {args.metric}: {best_score:.4f}")

    output_path = Path(args.output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    config = {
        "type": "per-class",
        "metric": args.metric,
        "thresholds": {str(k): float(v) for k, v in class_thresholds.items()},
        "score": float(best_score),
        "labels_sorted": labels_sorted,
    }
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    print(f"\nThreshold configuration saved to: {output_path}")


if __name__ == "__main__":
    main()
print('===== END {fname} =====')

===== START 08_threshold_optimization.py =====
Loaded 69 validation samples with 5 classes
Optimizing for: f1-macro

Finding per-class optimal thresholds...

Optimal per-class thresholds:
  belly_pain: 0.201
  burping: 0.204
  discomfort: 0.199
  hungry: 0.198
  tired: 0.200

Best f1-macro: 0.1724

Threshold configuration saved to: artifacts\model\threshold_config.json
===== END {fname} =====


C:\Users\Dell\AppData\Local\Temp\ipykernel_18212\1203179720.py:66: RuntimeWarning: invalid value encountered in divide
  (2.0 * precision * recall) / (precision + recall),
C:\Users\Dell\AppData\Local\Temp\ipykernel_18212\1203179720.py:66: RuntimeWarning: invalid value encountered in divide
  (2.0 * precision * recall) / (precision + recall),
C:\Users\Dell\AppData\Local\Temp\ipykernel_18212\1203179720.py:66: RuntimeWarning: invalid value encountered in divide
  (2.0 * precision * recall) / (precision + recall),


## 09_inference_with_thresholds.py


In [16]:
print('===== START 09_inference_with_thresholds.py =====')
#!/usr/bin/env python3
"""
Inference script with optional threshold adjustment for better minority class handling.
"""

import argparse
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


def load_threshold_config(config_file):
    """Load threshold configuration from JSON file."""
    if not Path(config_file).exists():
        return None
    
    with open(config_file) as f:
        config = json.load(f)
    return config


def apply_thresholds(y_proba, config):
    """Apply threshold configuration to predictions.
    
    Args:
        y_proba: Predicted probabilities (n_samples, n_classes)
        config: Threshold configuration dict
    
    Returns:
        y_pred: Predicted class indices
    """
    if config is None:
        return np.argmax(y_proba, axis=1)
    
    if config.get("type") == "global":
        return np.argmax(y_proba, axis=1)
    
    elif config.get("type") == "per-class":
        thresholds = {int(k): v for k, v in config["thresholds"].items()}
        y_pred = np.zeros(len(y_proba), dtype=np.int32)
        
        for i, proba_sample in enumerate(y_proba):
            ordered_classes = np.argsort(proba_sample)[::-1]
            chosen_class = int(ordered_classes[0])

            for class_idx in ordered_classes:
                class_idx = int(class_idx)
                if proba_sample[class_idx] >= thresholds.get(class_idx, 0.5):
                    chosen_class = class_idx
                    break

            y_pred[i] = chosen_class
        
        return y_pred
    
    return np.argmax(y_proba, axis=1)


def main():
    parser = argparse.ArgumentParser(description="Inference with threshold adjustment")
    parser.add_argument(
        "--model-path",
        type=str,
        default="artifacts/model/best_model.h5",
        help="Path to trained model",
    )
    parser.add_argument(
        "--threshold-config",
        type=str,
        default="artifacts/model/threshold_config.json",
        help="Path to threshold configuration JSON",
    )
    parser.add_argument(
        "--metadata-file",
        type=str,
        default="artifacts/model/metadata.json",
        help="Path to metadata JSON with labels",
    )
    parser.add_argument(
        "--test-predictions",
        type=str,
        default="artifacts/model/test_predictions.npy",
        help="Path to test predictions (for evaluation demo)",
    )
    parser.add_argument(
        "--test-labels",
        type=str,
        default="artifacts/model/test_labels.npy",
        help="Path to test labels (for evaluation demo)",
    )
    args = parser.parse_args(args=[])

    metadata_path = Path(args.metadata_file)
    if not metadata_path.exists():
        fallback_path = Path(args.model_path).resolve().parent / "training_info.json"
        if fallback_path.exists():
            metadata_path = fallback_path

    with metadata_path.open("r", encoding="utf-8") as f:
        metadata = json.load(f)

    if "labels_sorted" in metadata:
        labels_sorted = metadata["labels_sorted"]
    else:
        index_to_label = metadata.get("index_to_label", {})
        labels_sorted = [index_to_label[str(idx)] for idx in sorted(map(int, index_to_label.keys()))]

    n_classes = len(labels_sorted)

    # Load threshold config
    threshold_config = load_threshold_config(args.threshold_config)
    if threshold_config:
        print(f"Loaded threshold configuration from: {args.threshold_config}")
        print(f"  Type: {threshold_config['type']}")
        print(f"  Metric: {threshold_config['metric']}")
        if threshold_config["type"] == "global":
            print(f"  Global threshold: {threshold_config['threshold']:.3f}")
        else:
            print(f"  Per-class thresholds:")
            for class_idx, thresh in sorted(threshold_config["thresholds"].items()):
                print(f"    {labels_sorted[int(class_idx)]}: {thresh:.3f}")
    else:
        print("No threshold configuration found, using default (argmax)")

    # Demo: Load test predictions and evaluate
    if Path(args.test_predictions).exists():
        print("\n" + "=" * 60)
        print("EVALUATION ON TEST SET")
        print("=" * 60)

        test_predictions = np.load(args.test_predictions)
        test_labels = np.load(args.test_labels)

        # Get predictions with thresholds
        y_pred = apply_thresholds(test_predictions, threshold_config)

        # Accuracy
        accuracy = np.mean(y_pred == test_labels)
        print(f"\nAccuracy: {accuracy:.4f}")

        # Classification report
        print("\nClassification Report:")
        print(
            classification_report(
                test_labels,
                y_pred,
                target_names=labels_sorted,
                zero_division=0,
            )
        )

        # Confusion matrix
        print("\nConfusion Matrix:")
        cm = confusion_matrix(test_labels, y_pred)
        print(cm)


if __name__ == "__main__":
    main()
print('===== END {fname} =====')

===== START 09_inference_with_thresholds.py =====
Loaded threshold configuration from: artifacts/model/threshold_config.json
  Type: per-class
  Metric: f1-macro
  Per-class thresholds:
    belly_pain: 0.201
    burping: 0.204
    discomfort: 0.199
    hungry: 0.198
    tired: 0.200

EVALUATION ON TEST SET

Accuracy: 0.3913

Classification Report:
              precision    recall  f1-score   support

  belly_pain       0.00      0.00      0.00         2
     burping       0.00      0.00      0.00         1
  discomfort       0.06      0.25      0.09         4
      hungry       0.96      0.45      0.61        58
       tired       0.00      0.00      0.00         4

    accuracy                           0.39        69
   macro avg       0.20      0.14      0.14        69
weighted avg       0.81      0.39      0.52        69


Confusion Matrix:
[[ 0  1  0  1  0]
 [ 0  0  1  0  0]
 [ 0  3  1  0  0]
 [ 3 14 15 26  0]
 [ 0  3  1  0  0]]
===== END {fname} =====


## Final Evaluation Matrices And Metrics

In [17]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

model_dir = Path('artifacts/model').resolve()
print('Model dir:', model_dir)

info_path = model_dir / 'training_info.json'
per_class_path = model_dir / 'per_class_metrics.csv'
threshold_path = model_dir / 'threshold_config.json'

if info_path.exists():
    info = json.loads(info_path.read_text(encoding='utf-8'))
    print('\n=== SCALAR METRICS ===')
    print('test_loss:', info.get('test_loss'))
    print('test_accuracy:', info.get('test_accuracy'))

    print('\n=== CONFUSION MATRIX ===')
    cm = np.array(info.get('confusion_matrix', []))
    print(cm)

    print('\n=== CLASSIFICATION REPORT (TABLE) ===')
    report = info.get('classification_report', {})
    rows = []
    for k, v in report.items():
        if isinstance(v, dict):
            row = {'label': k}
            row.update(v)
            rows.append(row)
    if rows:
        print(pd.DataFrame(rows).to_string(index=False))
    else:
        print('No classification report rows found.')
else:
    print('training_info.json not found')

if per_class_path.exists():
    print('\n=== PER-CLASS METRICS CSV ===')
    print(pd.read_csv(per_class_path).to_string(index=False))
else:
    print('per_class_metrics.csv not found')

if threshold_path.exists():
    print('\n=== THRESHOLD CONFIG ===')
    print(json.dumps(json.loads(threshold_path.read_text(encoding='utf-8')), indent=2))
else:
    print('threshold_config.json not found')


Model dir: D:\IISc\sem2\EdgeAI\Project\proj-v2\cry_reason_pipeline\artifacts\model

=== SCALAR METRICS ===
test_loss: 0.9356445074081421
test_accuracy: 0.7536231875419617

=== CONFUSION MATRIX ===
[[ 2  0  0  0  0]
 [ 0  1  0  0  0]
 [ 0  0  0  4  0]
 [ 2  3  1 49  3]
 [ 1  0  0  3  0]]

=== CLASSIFICATION REPORT (TABLE) ===
       label  precision   recall  f1-score  support
  belly_pain   0.400000 1.000000  0.571429      2.0
     burping   0.250000 1.000000  0.400000      1.0
  discomfort   0.000000 0.000000  0.000000      4.0
      hungry   0.875000 0.844828  0.859649     58.0
       tired   0.000000 0.000000  0.000000      4.0
   macro avg   0.305000 0.568966  0.366216     69.0
weighted avg   0.750725 0.753623  0.744964     69.0

=== PER-CLASS METRICS CSV ===
     label  precision   recall       f1  support
belly_pain      0.400 1.000000 0.571429        2
   burping      0.250 1.000000 0.400000        1
discomfort      0.000 0.000000 0.000000        4
    hungry      0.875 0.844828